In [ ]:
from pathlib import Path

import numpy as np
import torch
import xarray as xr
from dotenv import load_dotenv
from earth2studio.data import GFS, prep_data_array
from earth2studio.models.dx import CorrDiffTaiwan
from earth2studio.utils.coords import map_coords
from earth2studio.utils.time import to_time_array

In [ ]:
load_dotenv()  # TODO: make common example prep function

# Create CorrDiff model
package = CorrDiffTaiwan.load_default_package()
corrdiff = CorrDiffTaiwan.load_model(package)

# Create the data source
data = GFS()

# Create the IO handler, store in memory
# io = ZarrBackend()

In [ ]:
datapath = Path.home() / "ml-ds_data" / "cwa_dataset_storm.zarr"

In [ ]:
time_string = "2021-09-02T00:00:00"

In [ ]:
ds = xr.open_zarr(datapath, consolidated=False)

In [ ]:
ds

In [ ]:
ds.cwb_center.cwb_variable.values

In [ ]:
u_target = ds.sel(time=time_string).cwb[2, :, :]

In [ ]:
v_target = ds.sel(time=time_string).cwb[3, :, :]

In [ ]:
time, number_of_samples = [time_string], 1

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
corrdiff = corrdiff.to(device)
corrdiff.number_of_samples = number_of_samples
time = to_time_array(time)

In [ ]:
x, coords = prep_data_array(
    data(time, corrdiff.input_coords()["variable"]), device=device
)
x, coords = map_coords(x, coords, corrdiff.input_coords())

In [ ]:
corrdiff.input_coords()["variable"]

In [ ]:
y_pred, coords_out = corrdiff(x, coords)

In [ ]:
coords_out["variable"]

In [ ]:
y_pred.shape

In [ ]:
u_x = x[0, 10, :, :].detach().cpu().numpy()
v_x = x[0, 11, :, :].detach().cpu().numpy()
lat_x = coords["lat"]
lon_x = coords["lon"]
lat_x = np.broadcast_to(lat_x[:, np.newaxis], u_x.shape)
lon_x = np.broadcast_to(lon_x[np.newaxis, :], u_x.shape)

In [ ]:
input_ds = xr.Dataset(
    data_vars={
        "u_x": (("y", "x"), u_x),
        "v_x": (("y", "x"), v_x),
    },
    coords={
        "lat": (("y", "x"), lat_x),
        "lon": (("y", "x"), lon_x),
        "time": time[0],
    },
    attrs={"description": "Input near-surface wind components from GFS"},
)

In [ ]:
u_pred = y_pred[0, 0, 2, :, :].detach().cpu().numpy()
v_pred = y_pred[0, 0, 3, :, :].detach().cpu().numpy()
lat_pred = coords_out["lat"]
lon_pred = coords_out["lon"]

In [ ]:
pred_ds = xr.Dataset(
    data_vars={
        "u_pred": (("y", "x"), u_pred),
        "v_pred": (("y", "x"), v_pred),
    },
    coords={
        "lat": (("y", "x"), lat_pred),
        "lon": (("y", "x"), lon_pred),
        "time": time[0],
    },
    attrs={"description": "CorrDiff predicted near-surface wind components"},
)

In [ ]:
input_ds["u_x"].plot(figsize=(12, 10))

In [ ]:
u_target.plot(figsize=(12, 10))

In [ ]:
pred_ds["u_pred"].plot(figsize=(12, 10))

In [ ]:
input_ds["v_x"].plot(figsize=(12, 10))

In [ ]:
v_target.plot(figsize=(12, 10))

In [ ]:
pred_ds["v_pred"].plot(figsize=(12, 10))

In [ ]:
def compute_divergence(u, v, lat, lon):
    """Compute horizontal divergence on a general curvilinear lat/lon grid."""
    R_earth = 6371000.0  # Earth radius in meters

    u = np.asarray(u)
    v = np.asarray(v)
    lat = np.asarray(lat)
    lon = np.asarray(lon)

    if u.shape != v.shape:
        raise ValueError(f"u and v must have the same shape, got {u.shape} and {v.shape}")

    if lat.ndim == 1 and lon.ndim == 1:
        lon_2d, lat_2d = np.meshgrid(lon, lat)
    elif lat.ndim == 2 and lon.ndim == 2:
        if lat.shape != lon.shape:
            raise ValueError(f"lat and lon 2D arrays must have same shape, got {lat.shape} and {lon.shape}")
        lat_2d, lon_2d = lat, lon
    else:
        raise ValueError("lat and lon must both be 1D arrays or both be 2D arrays")

    if u.shape != lat_2d.shape:
        raise ValueError(
            f"u/v shape {u.shape} is incompatible with lat/lon shape {lat_2d.shape}"
        )

    # Convert to radians; unwrap longitude to avoid artificial jumps at the dateline.
    lat_rad = np.radians(lat_2d)
    lon_rad = np.unwrap(np.unwrap(np.radians(lon_2d), axis=1), axis=0)

    # Derivatives in index space (y=row, x=column).
    du_dy, du_dx = np.gradient(u)
    dv_dy, dv_dx = np.gradient(v)
    dlam_dy, dlam_dx = np.gradient(lon_rad)
    dphi_dy, dphi_dx = np.gradient(lat_rad)

    # Invert local Jacobian to map index-space derivatives to (lambda, phi) derivatives.
    det = dlam_dx * dphi_dy - dlam_dy * dphi_dx
    det = np.where(np.abs(det) < 1e-14, np.nan, det)

    du_dlam = (du_dx * dphi_dy - du_dy * dphi_dx) / det
    dv_dphi = (dlam_dx * dv_dy - dlam_dy * dv_dx) / det

    cos_lat = np.cos(lat_rad)
    cos_lat = np.where(np.abs(cos_lat) < 1e-12, np.nan, cos_lat)

    divergence = du_dlam / (R_earth * cos_lat) + dv_dphi / R_earth

    print(f"Divergence shape: {divergence.shape}")
    print(f"Divergence range: [{np.nanmin(divergence):.2e}, {np.nanmax(divergence):.2e}]")
    return divergence

In [ ]:
uv_x_div = compute_divergence(u_x, v_x, lat_x, lon_x)

In [ ]:
uv_pred_div = compute_divergence(u_pred, v_pred, lat_pred, lon_pred)

In [ ]:
u_y, v_y = u_target.values, v_target.values
lat_y = u_target.XLAT.values
lon_y = u_target.XLONG.values

In [ ]:
uv_y_div = compute_divergence(u_y, v_y, lat_y, lon_y)

In [ ]:
div_ds = xr.Dataset(
    data_vars={
        "uv_x_div": (("y_x", "x_x"), uv_x_div),
        "uv_pred_div": (("y_pred", "x_pred"), uv_pred_div),
        "uv_y_div": (("y_y", "x_y"), uv_y_div),
    },
    coords={
        "lat_x": (("y_x", "x_x"), lat_x),
        "lon_x": (("y_x", "x_x"), lon_x),
        "lat_pred": (("y_pred", "x_pred"), lat_pred),
        "lon_pred": (("y_pred", "x_pred"), lon_pred),
        "lat_y": (("y_y", "x_y"), lat_y),
        "lon_y": (("y_y", "x_y"), lon_y),
        "time": time[0],
    },
    attrs={"description": "Horizontal divergence fields from input, prediction, and target winds"},
)

In [ ]:
div_ds["uv_x_div"].plot(figsize=(12, 10))

In [ ]:
div_ds["uv_pred_div"].plot(figsize=(12, 10))

In [ ]:
div_ds["uv_y_div"].plot(figsize=(12, 10))